# Plugins
> Plugin discovery for stores

In [ ]:
#| default_exp plugins

In [ ]:
#| export
import pluggy
from pathlib import Path
import importlib.util
from memexcode.pluginspec import SessionStoreHookSpec, hookspec

In [ ]:
#| export
def init_store_plugins():
    '''Discover stores via setuptools entry points (memexcode.store).'''
    pm = pluggy.PluginManager('memexcode')
    pm.add_hookspecs(SessionStoreHookSpec)
    pm.load_setuptools_entrypoints('memexcode.store')
    stores = {}
    for result in pm.hook.get_store():
        if result:
            name, cls = result
            stores[name] = cls
    return pm, stores

In [ ]:
#| export
def init_store_plugins_from_dir(plugin_dir: Path):
    '''Discover stores by scanning a directory for .py files.'''
    pm = pluggy.PluginManager('memexcode')
    pm.add_hookspecs(SessionStoreHookSpec)
    plugin_dir = Path(plugin_dir)
    if plugin_dir.exists():
        for fpath in sorted(plugin_dir.glob('*.py')):
            if fpath.name.startswith('_'):
                continue
            spec = importlib.util.spec_from_file_location(fpath.stem, fpath)
            mod = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(mod)
            pm.register(mod)
    stores = {}
    for result in pm.hook.get_store():
        if result:
            name, cls = result
            stores[name] = cls
    return pm, stores